<a href="https://colab.research.google.com/github/abeeraz379/Logistic-Regression-Optimization-Stroke-data-/blob/main/stroke.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# import libraries

In [83]:
# import numpy ,pandas
# import test train split
# import logisticregression
# import Gridsearchcv
# import scaler
# import pipeline
# import onehot encoder
# import simple imbuter
# import confusion matrix , confusion report
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix,classification_report
import warnings
warnings.filterwarnings('ignore')

# load and inspect data

In [84]:
# load stroke data
df=pd.read_csv("/content/drive/MyDrive/AXSOSACADEMY/AXSOSACADEMY/02-IntroML/Week07/Data/stroke.csv")
df.head()

,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,1192,Female,31,0,0,No,Govt_job,Rural,70.66,27.2,never smoked,0
1,77,Female,13,0,0,No,children,Rural,85.81,18.6,Unknown,0
2,59200,Male,18,0,0,No,Private,Urban,60.56,33.0,never smoked,0
3,24905,Female,65,0,0,Yes,Private,Urban,205.77,46.0,formerly smoked,1
4,24257,Male,4,0,0,No,children,Rural,90.42,16.2,Unknown,0


In [85]:
# show info about df
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1137 entries, 0 to 1136
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 1137 non-null   int64  
 1   gender             1137 non-null   object 
 2   age                1137 non-null   object 
 3   hypertension       1137 non-null   int64  
 4   heart_disease      1137 non-null   int64  
 5   ever_married       1137 non-null   object 
 6   work_type          1137 non-null   object 
 7   Residence_type     1137 non-null   object 
 8   avg_glucose_level  1137 non-null   float64
 9   bmi                1085 non-null   float64
 10  smoking_status     1137 non-null   object 
 11  stroke             1137 non-null   int64  
dtypes: float64(2), int64(4), object(6)
memory usage: 106.7+ KB


In [86]:
# show statistical info about df
df.describe()

,id,hypertension,heart_disease,avg_glucose_level,bmi,stroke
count,1137.000000,1137.000000,1137.000000,1137.000000,1085.000000,1137.000000
mean,36750.933157,0.118734,0.068602,107.664002,29.198065,0.120493
std,21112.281253,0.323617,0.252887,47.618723,7.669615,0.325680
min,77.000000,0.000000,0.000000,55.270000,11.300000,0.000000
25%,17986.000000,0.000000,0.000000,77.600000,24.100000,0.000000
50%,37479.000000,0.000000,0.000000,91.820000,28.500000,0.000000
75%,55410.000000,0.000000,0.000000,113.850000,33.200000,0.000000
max,72918.000000,1.000000,1.000000,266.590000,64.400000,1.000000


In [87]:
# show statistical info about object features
df.describe(include="object")

,gender,age,ever_married,work_type,Residence_type,smoking_status
count,1137,1137,1137,1137,1137,1137
unique,3,84,2,5,2,4
top,Female,79,Yes,Private,Urban,never smoked
freq,642,26,769,672,587,416


In [88]:
# show values of each column
for col in df.columns:
  print(col)
  print(df[col].unique())

id
[ 1192    77 59200 ... 59749 12109  5731]
gender
['Female' 'Male' 'Other']
age
['31' '13' '18' '65' '4' '28' '64' '62' '26' '63' '50' '77' '1' '52' '24'
 '48' '45' '74' '3' '17' '54' '55' '67' '35' '38' '81' '34' '44' '79' '56'
 '70' '30' '39' '29' '80' '59' '51' '19' '43' '71' '0' '23' '53' '78' '66'
 '60' '76' '22' '6' '47' '2' '46' '11' '33' '37' '49' '61' '75' '40' '8'
 '57' '12' '21' '41' '58' '27' '20' '68' '42' '5' '69' '9' '36' '25' '82'
 '73' '32' '7' '16' '14' '72' '15' '10' '*82']
hypertension
[0 1]
heart_disease
[0 1]
ever_married
['No' 'Yes']
work_type
['Govt_job' 'children' 'Private' 'Self-employed' 'Never_worked']
Residence_type
['Rural' 'Urban']
avg_glucose_level
[ 70.66  85.81  60.56 ... 234.35  80.43 108.61]
bmi
[27.2 18.6 33.  46.  16.2 30.3 31.4 31.7 20.3 27.4 34.5 32.8 17.2 32.3
 26.1 32.2 29.4 26.8 21.7 18.3 22.9 20.8 27.  25.6 24.8 27.6 44.6 43.2
 33.2 30.7 37.6 35.5 25.2 31.1 36.  21.4 24.2 26.3 29.  28.4 37.8 30.2
 40.1 45.9 33.4 31.2 29.1 27.3 24.6 23.6 20.

# Clean Data

In [89]:
# check missing values
df.isna().sum()

,0
id,0
gender,0
age,0
hypertension,0
heart_disease,0
ever_married,0
work_type,0
Residence_type,0
avg_glucose_level,0
bmi,52


In [90]:
# check duplicates
df.duplicated().sum()

np.int64(0)

# Preprocessing

In [91]:
# define x and y
x=df.drop(columns="stroke")
y=df["stroke"]

In [113]:
y.value_counts()

,count
stroke,
0,1000
1,137


In [92]:
# split data
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=1)

In [93]:
# impute missing value with median
# build scaler
# build one hot encoder

num_cols = x_train.select_dtypes("number").columns
cat_cols = x_train.select_dtypes("object").columns
from sklearn.compose import ColumnTransformer
num_transformer = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
cat_transformer = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder())
preprocessor = ColumnTransformer([('num', num_transformer, num_cols), ('cat', cat_transformer, cat_cols)])


# Build Model

In [94]:
# build logistic regression moddel
lr=LogisticRegression(random_state=1)

In [95]:
# make pipeline
pipe=make_pipeline(preprocessor,lr)


In [96]:
# fir pipe on data
pipe.fit(x_train,y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('standardscaler',
                                                                   StandardScaler())]),
                                                  Index(['id', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('simpleimputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehotencoder',
                                                                   OneHotEncoder())]),
                                                  Index(['gender', 'age', 'ever_married', 'work_type', 'Residence_type',
       'smoking_status'],
      dtype='object'))])),
                ('logisticregression', LogisticRegression(random_state=1))])

# Evaluate

In [97]:
# evaluate model using confusion matrix and report
# print accuracy of the model
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))


[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285

accuracy Training: 0.8838028169014085
accuracy Testing: 0.8666666666666667


# Tuning to Get better results

In [98]:
# tunning the model using gridsearchcv
pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('num',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['id', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi'], dtype='object')),
                                   ('cat',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='most_frequent')),
                                                    ('onehotencoder',
                                                     OneHotEncoder())]),
                                    Index(['gender', 'age', 'ever_married', 'work_type', 'Residence_type',
          'smokin

# Change Values of Parameters to get the best_params_

## C

In [99]:
# try diffrent values of 'logisticregression__C' to get the best_params_
param_grid={"logisticregression__C":[0.001,0.01,0.1, 1, 10, 100, 1000]}
grid=GridSearchCV(pipe,param_grid,cv=5)
grid.fit(x_train,y_train)
# get the best value
grid.best_params_

{'logisticregression__C': 0.001}

In [100]:
# rewrite the model with best value for c
lr=LogisticRegression(random_state=1,C=0.001)
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))


[[250   0]
 [ 35   0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       250
           1       0.00      0.00      0.00        35

    accuracy                           0.88       285
   macro avg       0.44      0.50      0.47       285
weighted avg       0.77      0.88      0.82       285

accuracy Training: 0.8802816901408451
accuracy Testing: 0.8771929824561403


**The F1-Score increase alittle bit, but the results for class 1 are all zeros**

## solver

In [101]:
# # try diffrent values of 'logisticregression__solver' to get the best_params_
param_grid={"logisticregression__solver":["newton-cg", "lbfgs", "liblinear", "sag", "saga"]}
grid=GridSearchCV(pipe,param_grid,cv=5)
grid.fit(x_train,y_train)
# get the best value
grid.best_params_


{'logisticregression__solver': 'newton-cg'}

In [102]:

# build the new version with newton-cg
lr=LogisticRegression(random_state=1,solver="newton-cg")
# evaluate the model
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))


[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285

accuracy Training: 0.8838028169014085
accuracy Testing: 0.8666666666666667


**The reults is the same for the defult parameters**

##penalty

In [103]:
# try diffrent values of logisticregression__penalty and 'logisticregression__solver' to get the best_params_
param_grid = {
    'logisticregression__penalty': ['l1', 'l2'],
    'logisticregression__solver': ['liblinear', 'saga']
}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)

{'logisticregression__penalty': 'l1', 'logisticregression__solver': 'liblinear'}


In [104]:
# try the new values of parameters
lr = LogisticRegression(random_state=1, penalty='l1', solver='liblinear')
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[243   7]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.97      0.93       250
           1       0.30      0.09      0.13        35

    accuracy                           0.86       285
   macro avg       0.59      0.53      0.53       285
weighted avg       0.81      0.86      0.83       285



**the F1-Score Decrease**

In [105]:
# try the another values of parameters
lr = LogisticRegression(random_state=1, penalty='l2')
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285



## max_iter

In [106]:
# Change the max_iter
param_grid = {'logisticregression__max_iter': [10,20,50,100,200,300,400,500]}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)


{'logisticregression__max_iter': 10}


In [107]:
# try the new value of parameters
lr = LogisticRegression(random_state=1, max_iter=10)
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285



**The results didn't change, logisticregression__max_iter is Not The Most effect Parameter**

## class_weight

In [108]:
 # try diffrent values of logisticregression__class_weight to get the best_params_
param_grid = {'logisticregression__class_weight': ['balanced', None]}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)



{'logisticregression__class_weight': 'balanced'}


In [109]:
# try the new value of parameters
lr = LogisticRegression(random_state=1,class_weight='balanced')
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[193  57]
 [ 14  21]]
              precision    recall  f1-score   support

           0       0.93      0.77      0.84       250
           1       0.27      0.60      0.37        35

    accuracy                           0.75       285
   macro avg       0.60      0.69      0.61       285
weighted avg       0.85      0.75      0.79       285



**The weighted avg increases, However the F1-Score decreaes**

## l1_ratio

In [110]:
 # try diffrent values of logisticregression__l1_ratio to get the best_params_
param_grid = {'logisticregression__l1_ratio': [0.0, 0.25, 0.5, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]}
grid = GridSearchCV(pipe, param_grid)
grid.fit(x_train, y_train)
print(grid.best_params_)

{'logisticregression__l1_ratio': 0.0}


In [111]:
# try the new value
lr = LogisticRegression(random_state=1,l1_ratio=0.0)
pipe = make_pipeline(preprocessor, lr)
pipe.fit(x_train, y_train)
y_pred = pipe.predict(x_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[244   6]
 [ 32   3]]
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       250
           1       0.33      0.09      0.14        35

    accuracy                           0.87       285
   macro avg       0.61      0.53      0.53       285
weighted avg       0.82      0.87      0.83       285



**it Didn't Change**

# Result

**For this Data Set - > The moset effective parameter is c , then pantly and class_weight**

In [112]:
# Build The model
lr=LogisticRegression(random_state=1, penalty='l2',solver="newton-cg",C=0.001)
pipe=make_pipeline(preprocessor,lr)
# evaluate the new version of model
pipe.fit(x_train,y_train)
y_pred=pipe.predict(x_test)
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))
print("accuracy Training:",pipe.score(x_train,y_train))
print("accuracy Testing:",pipe.score(x_test,y_test))

[[250   0]
 [ 35   0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.93       250
           1       0.00      0.00      0.00        35

    accuracy                           0.88       285
   macro avg       0.44      0.50      0.47       285
weighted avg       0.77      0.88      0.82       285

accuracy Training: 0.8802816901408451
accuracy Testing: 0.8771929824561403
